# Reader note

This notebook is part of the `arXiv:2606.04091` reproduction workflow. It reproduces lower panel of `Figures 6, 7,and 8`, the daily modulation for different materials calculated as $f_{mod} = max|R(t)-<R>|/<R>$.
Set `DATA_ROOT` to the directory containing the generated HDF5 data. Old execution outputs are intentionally cleared for release.


In [ ]:
import os
import h5py
import numpy as np
import script_helpers.script_fmod as dm
import matplotlib.pyplot as plt

# Generated HDF5 data root. Override this without editing the notebook by setting
DATA_ROOT='/path/to/generated/data before starting Jupyter.'

In [ ]:
### Set the target materials to plot (CaWO4 SiO2, Al2O3)
Targets = ["Al2O3", "SiO2", "CaWO4"]

### Set the threshold value to plot
new_threshold = 0.001 # 1 meV, 20 meV etc. (in eV)

### Define mediator types to plot (light_hadrophilic, heavy_hadrophilic, light_dark_photon)
Mediators = ["light_dark_photon"] # Choose the mediator name you want to plot

### numerics used in calculations (standard, standard_q_parallel)
Numerics = "standard"

### Halo models used in calculations (SHM, TSA, EMP)
Mod = ["SHM", "TSA", "EMP"]

### Target mass to plot (in MeV)
target_mass = 0.01 # in MeV

In [ ]:
### Conservative Velocity parameters in standard prescription
V_0 = [220, 200, 280]
V_E = [232, 217, 246]
V_ESC = [544, 450, 600]
fid_vel = "220_232_544" # Central values of the velocity parameters in the standard prescription

### V_0 parameters for TSA and EMP in rms matching prescription
V_0_tsa_h = [268, 234, 381] # vesc =600
V_0_tsa_m = [280, 244, 398] # vesc =544
V_0_tsa_l = [306, 266, 430] # vesc =450
fid_velocity_tsa = "280_232_544"

V_0_emp_h = [100, 126, 326] # vesc =600
V_0_emp_m = [115, 154, 710] # vesc =544
V_0_emp_l = [187, 340, np.inf] # vesc =450
fid_velocity_emp = "154_232_544"

V_ESC_h= [600]
V_ESC_m= [544]
V_ESC_l= [450]

In [ ]:
# Initialize dictionary to store dmod_min and dmod_max for each (Target, Mediator, Model)
dmod_results = {}

for Target in Targets:
    # Change this to the path where you have the generated HDF5 files for daily modulation
    # Current path is set to the publicly available data repository for the paper
    data_dir = DATA_ROOT + f"/Manuscript_data/{Target}_daily_modulation/files"
    data_dir_rms = DATA_ROOT + f"/Manuscript_data/{Target}_daily_modulation/files_vrms_QQQ"
    

    for mediator_key in Mediators:

        # Generate file prefixes for each model (SHM, TSA, EMP)
        prefixes_SHM = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[0], V_0, V_E, V_ESC)
        prefixes_TSA = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[1], V_0, V_E, V_ESC)
        prefixes_EMP = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[2], V_0, V_E, V_ESC)

        prefixes_tsa_l = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[1], V_0_tsa_l, V_E, V_ESC_l)
        prefixes_tsa_m = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[1], V_0_tsa_m, V_E, V_ESC_m)
        prefixes_tsa_h = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[1], V_0_tsa_h, V_E, V_ESC_h)
        prefixes_tsa = prefixes_tsa_h + prefixes_tsa_m + prefixes_tsa_l

        prefixes_emp_l = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[2], V_0_emp_l, V_E, V_ESC_l)
        prefixes_emp_m = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[2], V_0_emp_m, V_E, V_ESC_m)
        prefixes_emp_h = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[2], V_0_emp_h, V_E, V_ESC_h)
        prefixes_emp = prefixes_emp_h + prefixes_emp_m + prefixes_emp_l

        # Load files
        files_SHM = [h5py.File(os.path.join(data_dir, prefix + '.hdf5'), 'r') for prefix in prefixes_SHM]
        files_TSA = [h5py.File(os.path.join(data_dir, prefix + '.hdf5'), 'r') for prefix in prefixes_TSA]
        files_EMP = [h5py.File(os.path.join(data_dir, prefix + '.hdf5'), 'r') for prefix in prefixes_EMP]

        files_tsa = [h5py.File(os.path.join(data_dir_rms, prefix + '.hdf5'), 'r') for prefix in prefixes_tsa]
        files_emp = [h5py.File(os.path.join(data_dir_rms, prefix + '.hdf5'), 'r') for prefix in prefixes_emp]

        fileSHM = f"{Target}_{mediator_key}_standard_{Mod[0]}_{fid_vel}.hdf5"
        fileTSA = f"{Target}_{mediator_key}_standard_{Mod[1]}_{fid_vel}.hdf5"
        fileEMP = f"{Target}_{mediator_key}_standard_{Mod[2]}_{fid_vel}.hdf5"

        filetsa = f"{Target}_{mediator_key}_standard_{Mod[1]}_{fid_velocity_tsa}.hdf5"
        fileemp = f"{Target}_{mediator_key}_standard_{Mod[2]}_{fid_velocity_emp}.hdf5"

        fileSHM_path = os.path.join(data_dir, fileSHM)
        fileTSA_path = os.path.join(data_dir, fileTSA)
        fileEMP_path = os.path.join(data_dir, fileEMP)

        filetsa_path = os.path.join(data_dir_rms, filetsa)
        fileemp_path = os.path.join(data_dir_rms, fileemp)


        # Extract shared data
        DM_mass = files_SHM[0]['particle_physics/dm_properties/mass_list'][()] * 1e-6
        mass_index = np.argmin(np.abs(DM_mass - target_mass))
        time = files_SHM[0]['particle_physics/times'][()]
        Threshold = files_SHM[0]['particle_physics/threshold'][()] #* 1e3  # Convert to meV
        energy_bin_width = files_SHM[0]['numerics/energy_bin_width'][()] #* 1e3  # Convert to meV

        DM_mass_rms = files_tsa[0]['particle_physics/dm_properties/mass_list'][()] * 1e-6
        mass_index_rms = np.argmin(np.abs(DM_mass_rms - target_mass))

        # Picking the energy threshold
        if len(Threshold) == 1:
            th = 0
        else:
            print(f'{Target} - {mediator_key}: Pick for which energy threshold dmod has to be calculated from : {Threshold}')

        threshold_run = Threshold[th]

        # Calculate dmods for each model type
        dmods_max_SHM, dmods_min_SHM = dm.calculate_dailymod_maxmin_from_diff(files_SHM, time, mass_index, th, new_threshold, threshold_run, energy_bin_width)
        dmods_max_TSA, dmods_min_TSA = dm.calculate_dailymod_maxmin_from_diff(files_TSA, time, mass_index, th, new_threshold, threshold_run, energy_bin_width)
        dmods_max_EMP, dmods_min_EMP = dm.calculate_dailymod_maxmin_from_diff(files_EMP, time, mass_index, th, new_threshold, threshold_run, energy_bin_width)

        dmods_max_tsa, dmods_min_tsa = dm.calculate_dailymod_maxmin_from_diff(files_tsa, time, mass_index_rms, th, new_threshold, threshold_run, energy_bin_width)
        dmods_max_emp, dmods_min_emp = dm.calculate_dailymod_maxmin_from_diff(files_emp, time, mass_index_rms, th, new_threshold, threshold_run, energy_bin_width)

        # Compute fiducial dmods from these single files
        with h5py.File(fileSHM_path, 'r') as f_shm:
            dmod_SHM = dm.calculate_dmod_from_diff(f_shm, time, mass_index, th, new_threshold, threshold_run, energy_bin_width)

        with h5py.File(fileTSA_path, 'r') as f_tsa:
            dmod_TSA = dm.calculate_dmod_from_diff(f_tsa, time, mass_index, th, new_threshold, threshold_run, energy_bin_width)

        with h5py.File(fileEMP_path, 'r') as f_emp:
            dmod_EMP = dm.calculate_dmod_from_diff(f_emp, time, mass_index, th, new_threshold, threshold_run, energy_bin_width)

        with h5py.File(filetsa_path, 'r') as f_tsa1:
            dmod_tsa = dm.calculate_dmod_from_diff(f_tsa1, time, mass_index_rms, th, new_threshold, threshold_run, energy_bin_width)

        with h5py.File(fileemp_path, 'r') as f_emp1:
            dmod_emp = dm.calculate_dmod_from_diff(f_emp1, time, mass_index_rms, th, new_threshold, threshold_run, energy_bin_width)

        # Save results in dictionary
        dmod_results[(Target, mediator_key, 'SHM')] = {
            'dmod_max': dmods_max_SHM,
            'dmod_min': dmods_min_SHM,
            'fiducial': dmod_SHM,
            'DM_mass': DM_mass,
            'time': time,
            'threshold_index': th,
        }

        # TSA
        dmod_results[(Target, mediator_key, 'TSA')] = {
            'dmod_max': dmods_max_TSA,
            'dmod_min': dmods_min_TSA,
            'fiducial': dmod_TSA,
            'DM_mass': DM_mass,
            'time': time,
            'threshold_index': th,
        }

        # EMP
        dmod_results[(Target, mediator_key, 'EMP')] = {
            'dmod_max': dmods_max_EMP,
            'dmod_min': dmods_min_EMP,
            'fiducial': dmod_EMP,
            'DM_mass': DM_mass,
            'time': time,
            'threshold_index': th,
        }

        # TSA_RMS
        dmod_results[(Target, mediator_key, 'TSA_RMS')] = {
            'dmod_max': dmods_max_tsa,
            'dmod_min': dmods_min_tsa,
            'fiducial': dmod_tsa,
            'DM_mass': DM_mass_rms,
            'time': time,
            'threshold_index': th,
        }

        # EMP_RMS
        dmod_results[(Target, mediator_key, 'EMP_RMS')] = {
            'dmod_max': dmods_max_emp,
            'dmod_min': dmods_min_emp,
            'fiducial': dmod_emp,
            'DM_mass': DM_mass_rms,
            'time': time,
            'threshold_index': th,
        }

        # Close files
        for f in files_SHM + files_TSA + files_EMP + files_tsa + files_emp:
            f.close()

In [ ]:
# === Setup ===
model_colors = {"SHM": "#56B4E9", "TSA": "#E69F00", "EMP": "#CC79A7", "TSA_RMS":"blue", "EMP_RMS":"green"}
target_colors = {"CaWO4": "green", "SiO2": "darkturquoise", "Al2O3": "magenta"}
linestyles_new = {"max": "dashed", "min": "dotted"}

## Making the matplotlib plots look nicer
settings = {
    # 'figure.constrained_layout.use': True,
    # 'mathtext.fontset': 'stix',
    # 'font.family': 'STIXGeneral',
    # LaTeX-like fonts
    'mathtext.fontset': 'cm',
    # 'font.family': 'serif',
    # 'font.serif': ['Computer Modern Roman'],
    # 'mathtext.fontset': 'dejavuserif',
    'font.family': 'DejaVu Serif',
    'font.size':16,
    # 'axes.labelsize': 'large',
    'lines.markersize': 5,
    'axes.linewidth':2.0,
    'xtick.major.size':8.0,
    'xtick.minor.size':4.0,
    'xtick.major.width':1.5,
    'xtick.minor.width':1.0,
    'xtick.direction':'in', 
    'xtick.minor.visible':True,
    'xtick.top':True,
    'ytick.major.size':8.0,
    'ytick.minor.size':4.0,
    'ytick.major.width':1.5,
    'ytick.minor.width':1.0,
    'ytick.direction':'in', 
    'ytick.minor.visible':True,
    'ytick.right':True,
    'contour.linewidth':3.0,
    'savefig.bbox': 'tight',
    'savefig.dpi': 200,
}

plt.rcParams.update(**settings) 

In [ ]:
# Select the mediator you want to plot
mediator_to_plot = Mediators[0]  # Choose the mediator name you want to plot if you list multiple mediators in the Mediators list. 

In [ ]:
### Plotting dmod bands for all Targets and one chosen mediator

plt.figure(figsize=(8, 6))

for Target in Targets:
    key_SHM = (Target, mediator_to_plot, 'SHM')
    key_TSA = (Target, mediator_to_plot, 'TSA')
    key_EMP = (Target, mediator_to_plot, 'EMP')

    key_TSA_RMS = (Target, mediator_to_plot, 'TSA_RMS')
    key_EMP_RMS = (Target, mediator_to_plot, 'EMP_RMS')

    SHM_data = dmod_results[key_SHM]
    TSA_data = dmod_results.get(key_TSA, None)
    EMP_data = dmod_results.get(key_EMP, None)

    TSA_RMS_data = dmod_results.get(key_TSA_RMS, None)
    EMP_RMS_data = dmod_results.get(key_EMP_RMS, None)

    time = SHM_data['time']
    mass = SHM_data['DM_mass'][mass_index]
    mass_rms = TSA_RMS_data['DM_mass'][mass_index_rms]

    # SHM band
    plt.fill_between(time, SHM_data['dmod_min'], SHM_data['dmod_max'],
                     color= target_colors[Target], alpha=0.15)

    # SHM curve
    plt.plot(time,
             SHM_data['fiducial'],
             color=target_colors[Target],
             label=f'{Target} fiducial')
        
    # TSA curve
    if TSA_RMS_data is not None:
        plt.plot(time,
             TSA_RMS_data['fiducial'],
             color=target_colors[Target],
             linestyle = linestyles_new["max"],
             linewidth=1)
        
    # EMP curve
    if EMP_RMS_data is not None:
        plt.plot(time,
             EMP_RMS_data['fiducial'],
             color=target_colors[Target],
             linestyle=linestyles_new["min"],
             linewidth=1)
    

# Formatting
plt.xlabel(r"Time (hrs)", fontsize=20)
plt.ylabel(rf"R / ⟨R⟩", fontsize=20)
plt.xlim(0, 23)
plt.ylim(0, 2)
plt.text(0.22, 0.90, f'$\omega_{{\mathrm{{min}}}}$={new_threshold*1e3:.0f} meV, $m_\chi$={mass* 1e3:.0f} keV', transform=plt.gca().transAxes, verticalalignment='top',
                      alpha=0.8, fontsize=20)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()